In [ ]:
#@title Alice and Bob keys with Encoding and Decoding
import random
from collections import Counter

def generate_random_array(size=1024, values=[0, 1, 2, 3]):
    """
    Generate an array of random numbers from specified values

    Args:
        size: Number of elements to generate
        values: List of possible values

    Returns:
        List of random numbers
    """
    return [random.choice(values) for _ in range(size)]

def transform_to_alice_final(pure_array):
    """
    Transform the pure array to alice_final array according to rules:
    - 0 → 0 (unchanged)
    - 1 → 1 (unchanged)
    - 2 → 0 (changed)
    - 3 → 1 (changed)

    Args:
        pure_array: Original array with values 0, 1, 2, 3

    Returns:
        Transformed array with values 0 and 1 only
    """
    mapping = {0: 0, 1: 1, 2: 0, 3: 1}
    return [mapping[value] for value in pure_array]

def transform_to_bob_trine(pure_array):
    """
    Transform the pure array to bob_trine array according to probabilistic rules:

    For original 0:
        - 0 with 0.66 probability
        - 1 with 0.17 probability
        - 2 with 0.17 probability

    For original 1:
        - 1 with 0.5 probability
        - 2 with 0.5 probability

    For original 2:
        - 0 with 0.33 probability
        - 1 with 0.34 probability
        - 2 with 0.33 probability

    For original 3:
        - 0 with 0.33 probability
        - 1 with 0.33 probability
        - 2 with 0.34 probability

    Args:
        pure_array: Original array with values 0, 1, 2, 3

    Returns:
        Transformed array with values 0, 1, 2
    """
    # Define probability distributions for each input value
    # Format: (probabilities, output_values)
    mapping_rules = {
        0: ([0.66, 0.17, 0.17], [0, 1, 2]),      # 0: 66% → 0, 17% → 1, 17% → 2
        1: ([0.5, 0.5], [1, 2]),                  # 1: 50% → 1, 50% → 2
        2: ([0.33, 0.34, 0.33], [0, 1, 2]),       # 2: 33% → 0, 34% → 1, 33% → 2
        3: ([0.33, 0.33, 0.34], [0, 1, 2])        # 3: 33% → 0, 33% → 1, 34% → 2
    }

    bob_trine_array = []
    for value in pure_array:
        probs, outcomes = mapping_rules[value]
        bob_trine_array.append(random.choices(outcomes, weights=probs)[0])

    return bob_trine_array

def transform_to_bob_final(bob_trine_array):
    """
    Transform the bob_trine array to bob_final array according to rules:
    - 0 → 0
    - 1 → 1
    - 2 → 0

    Args:
        bob_trine_array: Array with values 0, 1, 2

    Returns:
        Transformed array with values 0 and 1 only
    """
    mapping = {0: 0, 1: 1, 2: 0}
    return [mapping[value] for value in bob_trine_array]

def analyze_array(arr, array_name="Array"):
    """
    Analyze the generated array

    Args:
        arr: List of numbers to analyze
        array_name: Name of the array for display
    """
    # Basic statistics
    print(f"\n{'='*50}")
    print(f"{array_name} Analysis:")
    print(f"{'='*50}")
    print(f"Array length: {len(arr)}")
    if len(arr) > 0:
        print(f"First 20 elements: {arr[:20]}")
        print(f"Last 20 elements: {arr[-20:]}")

    # Count frequencies
    frequencies = Counter(arr)
    print(f"\nFrequency of each value:")
    for num in sorted(frequencies.keys()):
        count = frequencies[num]
        percentage = (count / len(arr)) * 100 if len(arr) > 0 else 0
        print(f"  {num}: {count} times ({percentage:.2f}%)")

    # Verify all numbers are from the expected set
    unique_values = set(arr)
    print(f"\nUnique values in array: {sorted(unique_values)}")
    return unique_values

def analyze_conditional_probabilities(pure_array, bob_trine_array):
    """
    Analyze the conditional probabilities of bob_trine given pure values

    Args:
        pure_array: Original array with values 0, 1, 2, 3
        bob_trine_array: Transformed array with values 0, 1, 2
    """
    print(f"\n{'='*50}")
    print("Conditional Probability Analysis (Bob Trine given Pure):")
    print(f"{'='*50}")

    # Create a mapping of pure value to list of bob_trine values
    conditional_counts = {0: Counter(), 1: Counter(), 2: Counter(), 3: Counter()}

    for pure, bob in zip(pure_array, bob_trine_array):
        conditional_counts[pure][bob] += 1

    # Calculate and display conditional probabilities
    expected_rules = {
        0: {0: 0.66, 1: 0.17, 2: 0.17},
        1: {1: 0.5, 2: 0.5},
        2: {0: 0.33, 1: 0.34, 2: 0.33},
        3: {0: 0.33, 1: 0.33, 2: 0.34}
    }

    for pure_val in sorted(conditional_counts.keys()):
        total = sum(conditional_counts[pure_val].values())
        print(f"\nGiven Pure = {pure_val}:")
        for bob_val in sorted(conditional_counts[pure_val].keys()):
            count = conditional_counts[pure_val][bob_val]
            prob = count / total if total > 0 else 0
            expected = expected_rules[pure_val].get(bob_val, 0)
            print(f"  P(Bob={bob_val} | Pure={pure_val}) = {count}/{total} = {prob:.4f} "
                  f"(expected: {expected:.4f})")

# Generate the pure array with values 0, 1, 2, 3
pure_array = generate_random_array(1024, values=[0, 1, 2, 3])

# Analyze the pure array
pure_unique = analyze_array(pure_array, "Pure Array (0, 1, 2, 3)")

# Verify pure array contains only 0, 1, 2, 3
expected_pure = {0, 1, 2, 3}
unexpected_pure = pure_unique - expected_pure
if unexpected_pure:
    print(f"WARNING: Pure array has unexpected values: {unexpected_pure}")
else:
    print("✓ Pure array contains only expected values {0, 1, 2, 3}")

# Transform to alice_final array
alice_final = transform_to_alice_final(pure_array)

# Analyze the alice_final array
alice_unique = analyze_array(alice_final, "Alice Final Array (0, 1 only)")

# Verify alice_final contains only 0 and 1
expected_alice = {0, 1}
unexpected_alice = alice_unique - expected_alice
if unexpected_alice:
    print(f"WARNING: Alice final array has unexpected values: {unexpected_alice}")
else:
    print("✓ Alice final array contains only expected values {0, 1}")

# Transform to bob_trine array
bob_trine = transform_to_bob_trine(pure_array)

# Analyze the bob_trine array
bob_unique = analyze_array(bob_trine, "Bob Trine Array (0, 1, 2)")

# Verify bob_trine contains only 0, 1, 2
expected_bob = {0, 1, 2}
unexpected_bob = bob_unique - expected_bob
if unexpected_bob:
    print(f"WARNING: Bob trine array has unexpected values: {unexpected_bob}")
else:
    print("✓ Bob trine array contains only expected values {0, 1, 2}")

# Transform to bob_final array
bob_final = transform_to_bob_final(bob_trine)

# Analyze the bob_final array
bob_final_unique = analyze_array(bob_final, "Bob Final Array (0, 1 only - from Bob Trine)")

# Verify bob_final contains only 0 and 1
expected_bob_final = {0, 1}
unexpected_bob_final = bob_final_unique - expected_bob_final
if unexpected_bob_final:
    print(f"WARNING: Bob final array has unexpected values: {unexpected_bob_final}")
else:
    print("✓ Bob final array contains only expected values {0, 1}")

# Additional verification: Show the full transformation chain for first few elements with comparison
print(f"\n{'='*50}")
print("Complete Transformation Chain (first 16 elements of each array):")
print(f"{'='*50}")
print("Index | Pure | Alice Final | Bob Trine | Bob Final | Alice=Bob?")
print("-" * 70)
for i in range(16):
    alice_val = alice_final[i]
    bob_val = bob_final[i]
    match_status = "✓" if alice_val == bob_val else "✗"
    print(f"{i:5d} | {pure_array[i]:4d} | {alice_val:11d} | {bob_trine[i]:9d} | {bob_val:9d} | {match_status:9s}")

# Calculate overall match rate between Alice Final and Bob Final
alice_bob_matches = sum(1 for a, b in zip(alice_final, bob_final) if a == b)
alice_bob_match_percentage = (alice_bob_matches / len(alice_final)) * 100
print(f"\nAlice Final vs Bob Final Overall Match Rate: {alice_bob_matches}/{len(alice_final)} = {alice_bob_match_percentage:.2f}%")


def Alice_Encoding(plaintext, alice_val):
    """
    Encode a plaintext using Alice's value

    Args:
        plaintext: String of bits (e.g., "0011")
        alice_val: List of 1024 bits (0s and 1s)

    Returns:
        ciphertext: String of 1024 bits
    """
    # Convert alice_val list to string for easier manipulation
    alice_val_str = ''.join(str(bit) for bit in alice_val)

    # Generate extended_plaintext by copying each bit 256 times
    extended_plaintext = ""
    for bit in plaintext:
        extended_plaintext += bit * 256

    # XOR extended_plaintext with alice_val
    ciphertext = ""
    for i in range(1024):
        if extended_plaintext[i] == alice_val_str[i]:
            ciphertext += "0"
        else:
            ciphertext += "1"

    return ciphertext

def Bob_Decoding(ciphertext, bob_val):
    """
    Decode a ciphertext using Bob's value with majority voting

    Args:
        ciphertext: String of 1024 bits
        bob_val: List of 1024 bits (0s and 1s)

    Returns:
        plaintext_result: String of 4 bits
    """
    # Convert bob_val list to string for easier manipulation
    bob_val_str = ''.join(str(bit) for bit in bob_val)

    # XOR ciphertext with bob_val
    xor_result = ""
    for i in range(1024):
        if ciphertext[i] == bob_val_str[i]:
            xor_result += "0"
        else:
            xor_result += "1"

    # For each 256-bit block, determine majority bit
    plaintext_result = ""
    for i in range(0, 1024, 256):
        block = xor_result[i:i+256]
        zeros = block.count("0")
        ones = block.count("1")

        if zeros > ones:
            plaintext_result += "0"
        else:
            plaintext_result += "1"

    return plaintext_result

# Define the plaintext
plaintext = "0011"
print(f"\n{'='*50}")
print("ENCODING/DECODING WITH MAJORITY VOTING")
print(f"{'='*50}")
print(f"Original plaintext: {plaintext}")

# Use alice_final and bob_final as the keys
print(f"\nUsing Alice Final and Bob Final arrays as keys")
print(f"Alice key (first 20 bits): {alice_final[:20]}")
print(f"Bob key (first 20 bits): {bob_final[:20]}")

# Call Alice_Encoding to get ciphertext
ciphertext = Alice_Encoding(plaintext, alice_final)
print(f"\nCiphertext (first 20 bits): {ciphertext[:20]}...")
print(f"Ciphertext (last 20 bits): ...{ciphertext[-20:]}")

# Call Bob_Decoding to recover plaintext
plaintext_result = Bob_Decoding(ciphertext, bob_final)
print(f"\nDecoded plaintext result: {plaintext_result}")

# Check if they match
if plaintext == plaintext_result:
    print("✓ SUCCESS: Original plaintext and decoded plaintext match!")
else:
    print("✗ ERROR: Original plaintext and decoded plaintext do not match!")

# Analyze the bit agreement between Alice and Bob for each position in the 256-bit blocks
print(f"\n{'='*50}")
print("DETAILED MAJORITY VOTING ANALYSIS")
print(f"{'='*50}")

# For each of the 4 blocks (256 bits each), analyze the XOR result patterns
for block_num in range(4):
    start_idx = block_num * 256
    end_idx = start_idx + 256

    # Get the XOR result for this block after Bob's decoding step
    bob_xor_result = ""
    for i in range(start_idx, end_idx):
        if ciphertext[i] == str(bob_final[i]):
            bob_xor_result += "0"
        else:
            bob_xor_result += "1"

    zeros = bob_xor_result.count("0")
    ones = bob_xor_result.count("1")
    majority = "0" if zeros > ones else "1"
    expected_bit = plaintext[block_num]

    print(f"\nBlock {block_num+1} (bits {start_idx}-{end_idx-1}):")
    print(f"  Expected bit from plaintext: {expected_bit}")
    print(f"  Zeros in XOR result: {zeros}")
    print(f"  Ones in XOR result: {ones}")
    print(f"  Majority bit: {majority}")
    print(f"  Match: {'✓' if majority == expected_bit else '✗'}")

